# CS781 Phase 1 — Own BPE tokenizer + token cache

This notebook (CPU only, **Accelerator: None**) does three things:

1. **Trains our own byte-level BPE tokenizer from scratch** on a sample of the legal training corpus
   (pure Python, no tokenizer library — see the `LegalBPETokenizer` class below).
2. **Tokenizes the training / validation data** with it (same mix, split and seed as the model notebooks)
   and saves `train_tokens.npy`, `val_tokens.npy`, `meta.json`.
3. Saves the tokenizer next to the tokens in `/kaggle/working/cs781_tokens/tokenizer/`.

**How to use:** add the `HF` secret, Internet on, Accelerator **None**, then *Save Version → Save & Run All*.
When it finishes, create a Kaggle dataset from the `cs781_tokens` output folder and set `TOKEN_CACHE_DIR`
in the four `_ownBPE` model notebooks to its path. They load both the tokens and the tokenizer from there.


## 1. Hugging Face login

In [ ]:
import os
from huggingface_hub import login
from kaggle_secrets import UserSecretsClient

user_secrets = UserSecretsClient()
HF_TOKEN = user_secrets.get_secret("HF")
os.environ["HF_TOKEN"] = HF_TOKEN
login(token=HF_TOKEN)

print("Hugging Face login successful.")

## 2. Configuration
#### Data settings must match the model notebooks exactly (they are checked through `meta.json`).

In [ ]:
import os, json, random, time, gc
import numpy as np

HF_DATASET_REPO = "Exploration-Lab/CS-781-Capstone"
TRAIN_DIR = "Phase_1_Pretraining/Train"

TRAIN_MIX = {
    "train_supreme_court.json": 0.40,
    "train_high_courts.json":   0.30,
    "train_tribunals.json":     0.15,
    "train_acts.json":          0.15,
}
TRAIN_TOKENS = 50_000_000
VAL_TOKENS = 5_000_000
VAL_DOC_PROB = 0.05
SEED = 42

BPE_VOCAB_SIZE = 32_000
BPE_TRAIN_CHARS = 100_000_000
BPE_MIN_FREQUENCY = 2
TOKENIZER_NAME = f"legal-bpe-{BPE_VOCAB_SIZE // 1000}k"

EXISTING_TOKENIZER_DIR = None

N_WORKERS = os.cpu_count() or 2
TOKEN_DIR = "/kaggle/working/cs781_tokens"
TOKENIZER_DIR = os.path.join(TOKEN_DIR, "tokenizer")
os.makedirs(TOKEN_DIR, exist_ok=True)
random.seed(SEED); np.random.seed(SEED)
print("Tokenizer:", TOKENIZER_NAME, "| workers:", N_WORKERS)

## 3. Our BPE tokenizer (implemented from scratch)

In [ ]:
import heapq
import json
import os
import re
import time
from collections import Counter, defaultdict

PRETOKENIZE_PATTERN = (
    r"'(?:[sdmt]|ll|ve|re)| ?[^\W\d_]+| ?\d{1,3}| ?(?:[^\s\w]|_)+|\s+(?!\S)|\s+"
)
EOS_TOKEN = "<|endoftext|>"

class LegalBPETokenizer:
    def __init__(self, merges=None, name="legal-bpe"):
        self.name = name
        self.merges = [tuple(m) for m in (merges or [])]
        self._pattern = re.compile(PRETOKENIZE_PATTERN)
        self._build_tables()

    def _build_tables(self):
        self.ranks = {pair: i for i, pair in enumerate(self.merges)}
        self.vocab = {i: bytes([i]) for i in range(256)}
        for i, (a, b) in enumerate(self.merges):
            self.vocab[256 + i] = self.vocab[a] + self.vocab[b]
        self.eos_token = EOS_TOKEN
        self.eos_token_id = 256 + len(self.merges)
        self.pad_token = self.eos_token
        self.pad_token_id = self.eos_token_id
        self.model_max_length = 10**9
        self._cache = {}

    def __len__(self):
        return 256 + len(self.merges) + 1

    @property
    def vocab_size(self):
        return len(self)

    def pretokenize(self, text):
        return self._pattern.findall(text)

    @classmethod
    def train(cls, texts, vocab_size, name="legal-bpe", min_frequency=2, verbose=True):
        assert 258 <= vocab_size <= 65535, "vocab_size must fit in uint16"
        t0 = time.time()
        pattern = re.compile(PRETOKENIZE_PATTERN)

        word_counts = Counter()
        n_chars = 0
        for text in texts:
            n_chars += len(text)
            word_counts.update(pattern.findall(text))
        if verbose:
            print(f"[BPE] {n_chars:,} characters, {len(word_counts):,} unique words "
                  f"({time.time() - t0:.0f}s)")

        words = [list(w.encode("utf-8")) for w in word_counts]
        freqs = list(word_counts.values())
        del word_counts

        pair_counts = defaultdict(int)
        pair_where = defaultdict(set)
        for wi, (w, f) in enumerate(zip(words, freqs)):
            for a, b in zip(w, w[1:]):
                pair_counts[(a, b)] += f
                pair_where[(a, b)].add(wi)
        heap = [(-c, p) for p, c in pair_counts.items()]
        heapq.heapify(heap)

        n_merges = vocab_size - 257
        merges = []
        while len(merges) < n_merges and heap:
            neg_c, pair = heapq.heappop(heap)
            count = pair_counts.get(pair, 0)
            if count <= 0:
                continue
            if -neg_c != count:
                heapq.heappush(heap, (-count, pair))
                continue
            if count < min_frequency:
                if verbose:
                    print(f"[BPE] stopping early: best pair frequency {count} < {min_frequency}")
                break

            new_id = 256 + len(merges)
            merges.append(pair)
            a, b = pair
            changed = set()
            for wi in list(pair_where[pair]):
                w, f = words[wi], freqs[wi]
                for p in zip(w, w[1:]):
                    pair_counts[p] -= f
                    changed.add(p)
                new_w, i = [], 0
                while i < len(w):
                    if i + 1 < len(w) and w[i] == a and w[i + 1] == b:
                        new_w.append(new_id)
                        i += 2
                    else:
                        new_w.append(w[i])
                        i += 1
                words[wi] = new_w
                for p in zip(new_w, new_w[1:]):
                    pair_counts[p] += f
                    pair_where[p].add(wi)
                    changed.add(p)
            del pair_where[pair]
            pair_counts.pop(pair, None)
            changed.discard(pair)
            for p in changed:
                c = pair_counts.get(p, 0)
                if c > 0:
                    heapq.heappush(heap, (-c, p))
                else:
                    pair_counts.pop(p, None)
                    pair_where.pop(p, None)
            if verbose and (len(merges) % 2000 == 0):
                print(f"[BPE] {len(merges):,}/{n_merges:,} merges ({time.time() - t0:.0f}s)")

        tok = cls(merges, name=name)
        if verbose:
            print(f"[BPE] done: vocab size {len(tok):,} ({time.time() - t0:.0f}s)")
        return tok

    def _encode_word(self, word):
        ids = self._cache.get(word)
        if ids is not None:
            return ids
        ids = list(word.encode("utf-8"))
        ranks = self.ranks
        while len(ids) >= 2:
            best_rank, best_pair = None, None
            for p in zip(ids, ids[1:]):
                r = ranks.get(p)
                if r is not None and (best_rank is None or r < best_rank):
                    best_rank, best_pair = r, p
            if best_pair is None:
                break
            a, b = best_pair
            new_id = 256 + best_rank
            merged, i = [], 0
            while i < len(ids):
                if i + 1 < len(ids) and ids[i] == a and ids[i + 1] == b:
                    merged.append(new_id)
                    i += 2
                else:
                    merged.append(ids[i])
                    i += 1
            ids = merged
        if len(self._cache) > 2_000_000:
            self._cache.clear()
        self._cache[word] = ids
        return ids

    def encode(self, text, add_special_tokens=False):
        out = []
        for word in self._pattern.findall(text):
            out.extend(self._encode_word(word))
        return out

    def __call__(self, texts, add_special_tokens=False, **kwargs):
        if isinstance(texts, str):
            return {"input_ids": self.encode(texts)}
        return {"input_ids": [self.encode(t) for t in texts]}

    def decode(self, ids, skip_special_tokens=True, **kwargs):
        parts = []
        for i in ids:
            i = int(i)
            if i == self.eos_token_id:
                if not skip_special_tokens:
                    parts.append(self.eos_token.encode("utf-8"))
                continue
            b = self.vocab.get(i)
            if b is not None:
                parts.append(b)
        return b"".join(parts).decode("utf-8", errors="replace")

    def batch_decode(self, sequences, skip_special_tokens=True, **kwargs):
        return [self.decode(s, skip_special_tokens=skip_special_tokens) for s in sequences]

    def convert_ids_to_tokens(self, ids):
        return [self.vocab[i].decode("utf-8", errors="replace") if i in self.vocab else self.eos_token
                for i in ids]

    def save_pretrained(self, directory):
        os.makedirs(directory, exist_ok=True)
        path = os.path.join(directory, "legal_bpe_tokenizer.json")
        with open(path, "w") as f:
            json.dump({
                "type": "byte-level-bpe",
                "name": self.name,
                "pattern": PRETOKENIZE_PATTERN,
                "eos_token": self.eos_token,
                "eos_token_id": self.eos_token_id,
                "vocab_size": len(self),
                "merges": [list(m) for m in self.merges],
            }, f)
        return path

    @classmethod
    def from_pretrained(cls, directory):
        path = directory if directory.endswith(".json") else os.path.join(directory, "legal_bpe_tokenizer.json")
        with open(path) as f:
            data = json.load(f)
        assert data["pattern"] == PRETOKENIZE_PATTERN, "tokenizer was saved with a different pre-tokenisation pattern"
        tok = cls(data["merges"], name=data.get("name", "legal-bpe"))
        assert tok.eos_token_id == data["eos_token_id"]
        return tok

## 4. Download the training files

In [ ]:
from huggingface_hub import hf_hub_download

train_file_paths = {}
for fname, weight in TRAIN_MIX.items():
    if weight <= 0:
        continue
    train_file_paths[fname] = hf_hub_download(
        repo_id=HF_DATASET_REPO, filename=f"{TRAIN_DIR}/{fname}", repo_type="dataset", token=HF_TOKEN,
    )
    print(f"{fname}: {os.path.getsize(train_file_paths[fname]) / 1024**3:.2f} GB")

## 5. Document readers (same as the model notebooks)

In [ ]:
import gc

TOKENIZE_BATCH_DOCS = 64
LARGE_FILE_GB = 2.0
BYTES_PER_TOKEN = 4.0

def load_records(path):
    with open(path, "r", encoding="utf-8") as f:
        data = json.load(f)
    if isinstance(data, dict):
        data = list(data.values())
    return data

def record_text(record):
    if isinstance(record, dict):
        return record.get("text") or ""
    return str(record)

def iter_documents(path, rng, needed_tokens):
    size = os.path.getsize(path)
    if size < LARGE_FILE_GB * 1024**3:
        records = load_records(path)
        order = list(range(len(records)))
        rng.shuffle(order)
        print(f"  loaded {len(records):,} documents into memory")
        for i in order:
            yield record_text(records[i])
    else:
        try:
            import ijson
        except ImportError:
            import subprocess, sys
            subprocess.run([sys.executable, "-m", "pip", "install", "-q", "ijson"], check=True)
            import ijson
        keep_prob = min(1.0, 2.0 * needed_tokens / (size / BYTES_PER_TOKEN))
        print(f"  streaming {size / 1024**3:.1f} GB file, keeping each document with p={keep_prob:.3f}")
        with open(path, "rb") as f:
            head = f.read(4096).lstrip()[:1]
            f.seek(0)
            stream = ijson.items(f, "item") if head == b"[" else (v for _, v in ijson.kvitems(f, ""))
            for record in stream:
                if rng.random() < keep_prob:
                    yield record_text(record)

def batched(iterable, n):
    batch = []
    for item in iterable:
        batch.append(item)
        if len(batch) == n:
            yield batch
            batch = []
    if batch:
        yield batch

## 6. Train the BPE tokenizer
#### A sample of each source, in the same proportions as `TRAIN_MIX`.

In [ ]:
def bpe_training_texts():
    rng = random.Random(SEED + 1)
    total_weight = sum(w for w in TRAIN_MIX.values() if w > 0)
    for fname, weight in TRAIN_MIX.items():
        if weight <= 0:
            continue
        quota = int(BPE_TRAIN_CHARS * weight / total_weight)
        taken = 0
        docs = iter_documents(train_file_paths[fname], rng, needed_tokens=quota // 4)
        for text in docs:
            if not text:
                continue
            text = text[: quota - taken]
            taken += len(text)
            yield text
            if taken >= quota:
                break
        docs.close()
        print(f"  {fname}: {taken:,} characters used for BPE training")
        gc.collect()

if EXISTING_TOKENIZER_DIR:
    _cands = [EXISTING_TOKENIZER_DIR, os.path.join(EXISTING_TOKENIZER_DIR, "tokenizer")]
    _path = next((d for d in _cands if os.path.exists(os.path.join(d, "legal_bpe_tokenizer.json"))), None)
    assert _path, f"legal_bpe_tokenizer.json not found in {EXISTING_TOKENIZER_DIR} (or its tokenizer/ subfolder)"
    tokenizer = LegalBPETokenizer.from_pretrained(_path)
    TOKENIZER_NAME = tokenizer.name
    print("Reusing existing tokenizer from", _path, "->", TOKENIZER_NAME, "(BPE training skipped)")
else:
    tokenizer = LegalBPETokenizer.train(
        bpe_training_texts(), vocab_size=BPE_VOCAB_SIZE, name=TOKENIZER_NAME, min_frequency=BPE_MIN_FREQUENCY,
    )
tokenizer.save_pretrained(TOKENIZER_DIR)
EOS_ID = tokenizer.eos_token_id
print("Vocabulary size:", len(tokenizer), "| EOS id:", EOS_ID)

## 7. Quick look at the tokenizer

In [ ]:
samples = [
    "The appellant was convicted under Section 302 read with Section 34 IPC and sentenced to rigorous imprisonment.",
    "Learned counsel for the respondent submitted that the impugned order of the High Court is liable to be set aside.",
    "In W.P.(C) No. 12345/2019, a sum of Rs.5,00,000/- was directed to be paid within 30 days.",
]
for s in samples:
    ids = tokenizer.encode(s)
    assert tokenizer.decode(ids) == s, "round trip failed"
    print(f"{len(s.split())} words -> {len(ids)} tokens ({len(ids) / len(s.split()):.2f} tokens/word)")
    print("   ", tokenizer.convert_ids_to_tokens(ids))
longest = sorted(tokenizer.vocab.values(), key=len, reverse=True)[:20]
print("\nLongest tokens:", [t.decode("utf-8", "replace") for t in longest])

## 8. Tokenize train / validation data (parallel)
#### Same logic as the model notebooks: per-source quotas, split by document, EOS between documents.

In [ ]:
import multiprocessing as mp, itertools

def _encode_with_words(texts):
    return [(len(t.split()), tokenizer.encode(t)) for t in texts]

rng = random.Random(SEED)
total_weight = sum(w for w in TRAIN_MIX.values() if w > 0)
train_chunks, val_chunks = [], []
fertility_words = fertility_tokens = 0
ctx = mp.get_context("fork")
with ctx.Pool(N_WORKERS) as pool:
    for fname, weight in TRAIN_MIX.items():
        if weight <= 0:
            continue
        train_quota = int(TRAIN_TOKENS * weight / total_weight)
        val_quota = int(VAL_TOKENS * weight / total_weight)
        t0 = time.time()
        print(fname)
        documents = iter_documents(train_file_paths[fname], rng, train_quota + val_quota)
        n_train = n_val = n_docs = 0
        doc_batches = batched(documents, 64)
        done = False
        while not done:
            chunk = list(itertools.islice(doc_batches, N_WORKERS * 4))
            if not chunk:
                break
            for texts_ids in pool.map(_encode_with_words, chunk):
                for n_words, ids in texts_ids:
                    if not ids:
                        continue
                    n_docs += 1
                    fertility_words += n_words; fertility_tokens += len(ids)
                    ids.append(EOS_ID)
                    if rng.random() < VAL_DOC_PROB:
                        if n_val < val_quota:
                            ids = ids[:val_quota - n_val]
                            val_chunks.append(np.asarray(ids, dtype=np.uint16))
                            n_val += len(ids)
                    elif n_train < train_quota:
                        ids = ids[:train_quota - n_train]
                        train_chunks.append(np.asarray(ids, dtype=np.uint16))
                        n_train += len(ids)
                    if n_train >= train_quota and n_val >= val_quota:
                        done = True
                        break
                if done:
                    break
        documents.close()
        print(f"  {n_docs:,} docs used | train {n_train:,} / val {n_val:,} tokens | {time.time() - t0:.0f}s")
        if n_train < train_quota:
            print(f"  note: source exhausted before its train quota of {train_quota:,}")
        gc.collect()

train_tokens = np.concatenate(train_chunks)
val_tokens = np.concatenate(val_chunks)
del train_chunks, val_chunks
print(f"Train tokens: {len(train_tokens):,} | Validation tokens: {len(val_tokens):,}")
print(f"Average tokens per word on the corpus: {fertility_tokens / max(1, fertility_words):.3f}")

## 9. Save the token cache (+ tokenizer) and metadata

In [ ]:
token_meta = {
    "mix": TRAIN_MIX, "train_tokens": TRAIN_TOKENS, "val_tokens": VAL_TOKENS,
    "val_doc_prob": VAL_DOC_PROB, "seed": SEED, "tokenizer": TOKENIZER_NAME,
}
np.save(os.path.join(TOKEN_DIR, "train_tokens.npy"), train_tokens)
np.save(os.path.join(TOKEN_DIR, "val_tokens.npy"), val_tokens)
with open(os.path.join(TOKEN_DIR, "meta.json"), "w") as f:
    json.dump(token_meta, f, indent=2)
with open(os.path.join(TOKEN_DIR, "tokenizer_stats.json"), "w") as f:
    json.dump({"vocab_size": len(tokenizer), "bpe_train_chars": BPE_TRAIN_CHARS,
               "tokens_per_word": fertility_tokens / max(1, fertility_words)}, f, indent=2)
print("Saved to", TOKEN_DIR, "->", sorted(os.listdir(TOKEN_DIR)))
print("\nDecoded start of the training tokens:\n", tokenizer.decode(train_tokens[:120].tolist()))